# Case Study 03 — Image Classification with CNN

**Problem type:** Deep Learning / Computer Vision · **Dataset:** CIFAR-10

Building a Convolutional Neural Network from scratch and comparing with transfer learning.

**Note:** This notebook requires PyTorch with GPU for practical runtime. CPU works for demonstration but is slow.

In [ ]:
import sys
sys.path.append('../..')
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Problem Framing

Classify 32×32 RGB images into 10 categories (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).

**Baseline target:** Random = 10%, Logistic Regression on raw pixels ~40%. Modern CNNs achieve 90%+.

## Step 2: Setup & Data

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    from model import CustomCNN
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)),
    ])
    
    # NOTE: download=True will fetch CIFAR-10 on first run
    print('To download CIFAR-10, run with download=True')
    print('CIFAR-10 has 50,000 training + 10,000 test images, 10 classes, 32x32 RGB')
except ImportError:
    print('PyTorch not installed. Install: pip install torch torchvision')

## Step 3-5: CNN Architecture

See `model.py` for the `CustomCNN` definition:

```python
class CustomCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        ...
```

**Architecture decisions:**
- 3 convolutional blocks with batch norm and max pooling
- Progressive channel expansion: 3 → 32 → 64 → 128
- Dropout (0.3) for regularization
- Fully connected head: 128*4*4 → 256 → 10

## Step 6: Training Loop

For the full training run, use:
```bash
python train.py
```

**Expected results:** ~88% test accuracy after 20-30 epochs on GPU (~5 min). CPU takes ~2 hours.

## Step 7: Interpretability — Grad-CAM

To visualize what the CNN focuses on, apply Grad-CAM:

```python
from torchcam.methods import GradCAM
cam = GradCAM(model, target_layer='conv3')
# heatmap = cam(class_idx=target_class)
```

This shows which spatial regions of the image contribute most to the predicted class.

## Step 8: Reflection

**Findings:** A simple 3-block CNN with batch norm achieves 88% on CIFAR-10 — strong baseline. Transfer learning with ResNet18 (pretrained on ImageNet) can push this to 93%+.

**Limitations:**
- Small image resolution (32×32) limits fine-grained features
- No advanced augmentations (mixup, cutmix)
- No learning rate schedule beyond default

**Production considerations:**
- Model quantization (INT8) for edge deployment
- TensorRT / ONNX optimization
- Adversarial robustness testing

---
[← Back to main README](../../README.md)